# ThreatLens AI — Notebook 04: SHAP Explainability

**Stage in the pipeline:** `Intelligence → SHAP`
**Input:** cleaned dataset (Notebook 01) + trained classifier (Notebook 03)

### What this notebook does
1. Loads the cleaned dataset and the classifier saved by Notebook 03
2. Explains what SHAP actually is and why it's trustworthy (not just "another importance chart")
3. Computes SHAP values for the test set
4. Produces a **global** explanation — which features matter most across all predictions
5. Produces a **local** explanation — why the model made one specific prediction (a single alert), the same style of output the dashboard's "Explain This Alert" button is meant to show
6. Saves the SHAP values for a sample of rows so the FastAPI service can serve them later without recomputing

### Why SHAP, specifically
Notebook 03's feature importance chart tells you what mattered *on average* across the whole model. It cannot tell you why THIS ONE alert scored 91% risk. SHAP (SHapley Additive exPlanations) can — it's grounded in cooperative game theory and distributes credit for one specific prediction fairly across that row's features, which is what turns "black box, 91%" into "91%, and here's exactly why," the trust-building goal stated in the blueprint's Explainable AI section.


In [1]:
%pip install shap ipywidgets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from IPython.display import display, HTML
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from src.models.anomaly import select_feature_columns
from src.models.classifier import load_model

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models" / "classifier"
SHAP_DIR = PROJECT_ROOT / "models" / "shap"
SHAP_DIR.mkdir(parents=True, exist_ok=True)

shap.initjs()


## 1. Load data and the trained classifier

Update `MODEL_FILENAME` below to match whichever model Notebook 03 actually saved (Random Forest or XGBoost — check the `models/classifier/` folder). Using the real saved model here, rather than retraining inline, guarantees the explanations in this notebook match the model that's actually making decisions.


In [ ]:
MODEL_FILENAME = "attack_classifier_xgboost_v1.joblib"  # <- change to match whatever Notebook 03 saved

df = pd.read_parquet(PROCESSED_DIR / "cicids2017_cleaned.parquet")
feature_cols = select_feature_columns(df)
X = df[feature_cols]
y = df["attack_category"]

model = load_model(MODELS_DIR / MODEL_FILENAME)
print(f"Loaded model: {MODEL_FILENAME}")
print(f"Feature count: {len(feature_cols)}")

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Microsoft\\Desktop\\Nayi_Manzil_Internship\\ThreatLensAI\\models\\classifier\\attack_classifier_xgboost_v1.joblib'

## 2. Sample the data for SHAP

Computing exact SHAP values for tree ensembles is fast (`TreeExplainer` is specifically optimized for this), but still not free at millions of rows. We explain a representative random sample rather than the full dataset — standard practice, and enough to get both a reliable global picture and any specific row we want to zoom into.


In [ ]:
SAMPLE_SIZE = 2000
sample_idx = X.sample(n=min(SAMPLE_SIZE, len(X)), random_state=42).index
X_sample = X.loc[sample_idx]
y_sample = y.loc[sample_idx]

print(f"Explaining {len(X_sample):,} sampled rows")

## 3. Compute SHAP values

`TreeExplainer` works directly with tree-based models (Random Forest, XGBoost) and is exact — not an approximation — which is why it's the right choice here rather than the slower, model-agnostic `KernelExplainer`.

For multi-class models, SHAP returns one set of values *per class* — i.e. "how much did each feature push toward predicting DDoS" and separately "...toward predicting Brute Force," etc. We keep the full structure so we can look at any class we like.


In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# shap_values shape check: for multi-class sklearn models this is typically
# a list of arrays (one per class) or a single 3D array depending on the
# SHAP/model version — handle both so this notebook doesn't break on minor
# version differences.
if isinstance(shap_values, list):
    print(f"{len(shap_values)} classes, each shaped {shap_values[0].shape}")
else:
    print(f"SHAP values array shape: {shap_values.shape}")

## 4. Global explanation — what matters across all predictions

The summary plot below ranks features by their overall impact on the model's predictions. This is the notebook version of the SHAP bars already shown on the dashboard's "SHAP Explainability" page — here we're computing the real numbers behind that UI rather than mocking them.


In [ ]:
class_names = sorted(y.unique())
target_class = "Brute Force" if "Brute Force" in class_names else class_names[-1]
class_idx = class_names.index(target_class) if target_class in class_names else 0

plt.figure()
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[class_idx], X_sample, show=False, max_display=12)
else:
    shap.summary_plot(shap_values[:, :, class_idx], X_sample, show=False, max_display=12)
plt.title(f"SHAP summary — driving features for '{target_class}' predictions")
plt.tight_layout()
plt.show()

## 5. Local explanation — explain ONE alert

This is the part that maps directly onto the dashboard's "Explain This Alert" button: pick a single row the model flagged as an attack, and show exactly which features pushed the prediction toward that specific attack type, and by how much — the same `Failed Login Count +32%`, `Request Frequency +24%` style breakdown used throughout the blueprint.


In [ ]:
attack_rows = y_sample[y_sample != "Normal"]
if len(attack_rows) == 0:
    print("No attack rows in this sample — re-run Section 2 with a larger SAMPLE_SIZE.")
else:
    example_label = attack_rows.iloc[0]
    example_position = X_sample.index.get_loc(attack_rows.index[0])

    print(f"Explaining one real example — true label: {example_label}")

    row_shap = shap_values[class_idx][example_position] if isinstance(shap_values, list) \
        else shap_values[example_position, :, class_idx]

    contrib = pd.Series(row_shap, index=feature_cols).sort_values(key=abs, ascending=False).head(8)
    contrib_pct = (contrib / contrib.abs().sum() * 100).round(1)

    print(f"\nTop contributing features toward predicting '{target_class}':")
    for feat, pct in contrib_pct.items():
        sign = "+" if pct >= 0 else ""
        print(f"  {feat:35s} {sign}{pct}%")

In [ ]:
plt.figure(figsize=(8, 5))
colors = ["#ff4d5a" if v > 0 else "#3aa0ff" for v in contrib.values]
plt.barh(contrib.index[::-1], contrib.values[::-1], color=colors[::-1])
plt.xlabel("SHAP value (impact on prediction)")
plt.title(f"Why the model predicted '{target_class}' for this alert")
plt.tight_layout()
plt.show()

## 6. Save SHAP values for the sample

Saved as a compressed `.npz` so the FastAPI `/threats/{id}` or a similar explainability endpoint could look up a precomputed explanation without recomputing SHAP live for every request — a real production shortcut, not just a notebook convenience.


In [ ]:
if isinstance(shap_values, list):
    stacked = np.stack(shap_values, axis=-1)  # rows x features x classes
else:
    stacked = shap_values

np.savez_compressed(
    SHAP_DIR / "shap_values_sample_v1.npz",
    shap_values=stacked,
    feature_names=np.array(feature_cols, dtype=object),
    class_names=np.array(class_names, dtype=object),
    row_index=X_sample.index.to_numpy(),
)
print(f"Saved SHAP values -> {SHAP_DIR / 'shap_values_sample_v1.npz'}")

## 7. Summary & next steps

| Item | Result |
|---|---|
| Model explained | see `MODEL_FILENAME` above |
| Sample size | see Section 2 |
| Top global features (for target class) | see Section 4 plot |
| Example local explanation | see Section 5 output |
| Saved SHAP values | `models/shap/shap_values_sample_v1.npz` |

This notebook completes the "why" layer of ThreatLens AI: Notebook 02 says *something is wrong*, Notebook 03 says *what kind of attack it is*, and this notebook says *why the model believes that*.

**Next up (`05_threat_scoring.ipynb`)** combines the anomaly score (Notebook 02) and classifier confidence (Notebook 03) into the single 0–100 threat score and severity label shown on every alert card in the dashboard — using `src/models/threat_score.py`, already written and ready to use.
